# 02 · Comprensión de los datos de BCS (fotos y bitácora)

Fase 2 de CRISP-DM para los datos que tienen BCS: las fotos de la visita al CAETEC del 14/10/2026, la bitácora de los expertos (`etiquetas_bsc.xlsx`) y el dataset `vms-3.csv`. Los CSV de ordeño solo se usan para cruzar los IDs de las vacas; su análisis está en `01_ordenos.ipynb`.

El notebook **solo lee** los datos: no borra, mueve ni convierte nada. La limpieza y la transformación corresponden a la fase 3 (Preparación de los datos).

Las celdas **Resultados** y **Observaciones** se llenan a mano después de correr el notebook, en los lugares marcados con [COMPLETAR].

## Configuración

Definimos dónde están los datos. No están en el repositorio: cada integrante los descarga del bucket de S3 en la carpeta `data/` de la raíz del repo, que está en `.gitignore`. `RAIZ` apunta a esa carpeta con una ruta relativa al notebook (`../../data`), así que funciona en cualquier computadora sin cambiar nada.

También activamos `pillow-heif`, la librería que permite leer fotos HEIC. Si no está instalada, esas fotos aparecen como no leídas en 2.2.

La celda imprime `OK` o `NO EXISTE` para cada ruta. Si alguna dice `NO EXISTE`, revisa que los datos estén en `data/` con la estructura del README.

In [ ]:
from pathlib import Path
from collections import Counter
import datetime as dt
import hashlib
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from openpyxl import load_workbook
from PIL import Image

try:
    import pillow_heif
    pillow_heif.register_heif_opener()
    LEE_HEIC = True
except ImportError:
    LEE_HEIC = False

# Carpeta data/ en la raíz del repositorio (está en .gitignore). Ruta relativa a src/notebooks/.
RAIZ = Path("../../data")
DIR_FOTOS = RAIZ / "fotosextraidas"
BITACORA = RAIZ / "etiquetas_bsc.xlsx"
DIR_CSV = RAIZ / "csv"
VMS3 = DIR_CSV / "vms-3.csv"

# Las gráficas se guardan junto al notebook
FIGURAS = Path("figuras")
FIGURAS.mkdir(exist_ok=True)

pd.set_option("display.max_rows", 120)
for p in (DIR_FOTOS, BITACORA, DIR_CSV, VMS3):
    print("OK       " if p.exists() else "NO EXISTE", p)
print("Lectura de HEIC:", "sí" if LEE_HEIC else "no (instala pillow-heif)")

### Funciones auxiliares

- `sha256()`: calcula la huella del contenido de cada archivo. **Decidimos detectar duplicados por contenido y no por nombre**, porque una misma foto puede tener nombres distintos (`IMG_0530.HEIC` e `IMG_0530(1).HEIC`) y dos fotos distintas pueden llamarse igual (cada teléfono numera sus fotos por su cuenta).
- `metadatos()`: lee de cada foto el tamaño, la cámara, la fecha, la orientación y si trae mapa de profundidad. Solo lee el encabezado del archivo, sin decodificar los píxeles, para que sea rápido.
- `norm()`: pasa un nombre de columna a minúsculas y sin acentos ni símbolos ("ID - VACA" → "idvaca"). Así encontramos las columnas aunque cambien los espacios o los acentos.
- `texto()`: convierte una celda de Excel a texto (1221.0 → "1221"; una celda con formato de hora → "HH MM").
- `clave()` y `ordenar_ids()`: ordenan los IDs como números, para que 1204 quede antes de 91549.

In [ ]:
def sha256(ruta, bloque=1 << 20):
    """Huella del contenido del archivo: dos archivos con la misma huella son idénticos."""
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for b in iter(lambda: f.read(bloque), b""):
            h.update(b)
    return h.hexdigest()


def metadatos(ruta):
    """Tamaño, cámara, fecha, orientación y profundidad de una foto (sin decodificar los píxeles)."""
    r = {"ancho": None, "alto": None, "camara": "", "fecha": "", "orientacion": None,
         "profundidad": False, "error": ""}
    try:
        with Image.open(ruta) as im:
            exif = im.getexif()
            r["ancho"], r["alto"] = im.size
            r["camara"] = f"{exif.get(0x010F, '')} {exif.get(0x0110, '')}".strip()
            fecha = exif.get_ifd(0x8769).get(0x9003) or exif.get(0x0132) or ""
            r["fecha"] = str(fecha)[:10].replace(":", "-")
            r["orientacion"] = exif.get(0x0112)
            r["profundidad"] = bool(im.info.get("depth_images"))
    except Exception as e:
        r["error"] = str(e)[:120]
    return r


def norm(texto):
    """Nombre de columna en minúsculas, sin acentos ni símbolos ('ID - VACA' -> 'idvaca')."""
    t = str(texto).lower().translate(str.maketrans("áéíóúüñ", "aeiouun"))
    return re.sub(r"[^a-z0-9]", "", t)


def texto(v):
    """Celda de Excel a texto (1221.0 -> '1221', una hora -> 'HH MM')."""
    if v is None:
        return ""
    if isinstance(v, float) and v.is_integer():
        return str(int(v))
    if isinstance(v, (dt.time, dt.datetime)):
        return v.strftime("%H %M")
    return str(v).strip()


def clave(x):
    """Ordena los IDs como números cuando lo son."""
    return (0, int(x)) if str(x).isdigit() else (1, str(x))


def ordenar_ids(ids):
    return sorted(ids, key=clave)

## 2.1 Recolección de los datos

### Inventario de las fuentes

Recorremos las carpetas de fotos y, por cada imagen, guardamos su vaca, la descarga de la que viene, su tamaño, su huella y sus metadatos.

Decisiones:
- **La vaca de cada foto es el nombre de la carpeta que la contiene**, porque así estaban organizadas en Drive (una carpeta por vaca).
- Solo contamos archivos de imagen (HEIC, HEIF, JPG, JPEG y PNG) e ignoramos los archivos ocultos del sistema.
- **Las vacas de ordeño se toman del nombre de cada CSV**, porque cada archivo se llama como el ID de su vaca. Aquí no se lee su contenido.

In [ ]:
EXT_FOTO = {".heic", ".heif", ".jpg", ".jpeg", ".png"}
filas = []
for p in sorted(DIR_FOTOS.rglob("*")):
    if p.is_file() and p.suffix.lower() in EXT_FOTO and not p.name.startswith("."):
        rel = p.relative_to(DIR_FOTOS)
        filas.append({"id_vaca": p.parent.name.strip(), "descarga": rel.parts[0],
                      "archivo": p.name, "ruta": str(rel), "ext": p.suffix.lower(),
                      "bytes": p.stat().st_size, "sha256": sha256(p)})
fotos = pd.DataFrame(filas)
fotos = pd.concat([fotos, pd.DataFrame([metadatos(DIR_FOTOS / r) for r in fotos["ruta"]])], axis=1)

csv_ordeno = sorted(p for p in DIR_CSV.rglob("*.csv") if p.stem.strip().isdigit())
ids_ordeno = {p.stem.strip() for p in csv_ordeno}

fuentes = pd.DataFrame([
    {"fuente": "Fotos (fotosextraidas/)", "archivos": len(fotos), "MB": fotos["bytes"].sum() / 1e6},
    {"fuente": "Bitácora de expertos (etiquetas_bsc.xlsx)", "archivos": 1, "MB": BITACORA.stat().st_size / 1e6},
    {"fuente": "Dataset VMS 3 (vms-3.csv)", "archivos": 1, "MB": VMS3.stat().st_size / 1e6},
    {"fuente": "CSV de ordeño (uno por vaca)", "archivos": len(csv_ordeno),
     "MB": sum(p.stat().st_size for p in csv_ordeno) / 1e6},
]).round({"MB": 2})
display(fuentes)

**Resultados:** Recibimos [COMPLETAR] archivos de fotos ([COMPLETAR] MB), la bitácora de expertos ([COMPLETAR] MB), el dataset `vms-3.csv` y [COMPLETAR] CSV de ordeño, uno por vaca.

### Bitácora de expertos

La bitácora es un Excel compartido y sus encabezados no están en la primera fila. **En lugar de suponer una fila fija, buscamos en las primeras 40 filas de cada hoja la que tenga columnas que empiecen con "ID" y "BCS".** Guardamos el número real de fila de Excel (`fila_excel`) para poder ubicar cada evaluación en el archivo original.

Decisiones:
- Usamos solo el ID, la hora y el BCS. Las columnas "Foto" y "Carpeta" son ligas de Drive y no aportan datos para el análisis.
- El BCS se convierte a número aceptando punto o coma decimal. Si una celda no es número, queda vacía y se reporta en 2.4.

In [ ]:
# Bitácora: se busca la fila que tiene los encabezados "ID" y "BCS" (puede no ser la primera)
wb = load_workbook(BITACORA, read_only=True, data_only=True)
hoja, fila_enc = None, None
for ws in wb.worksheets:
    for i, fila in enumerate(ws.iter_rows(min_row=1, max_row=40, values_only=True), start=1):
        nombres = [norm(v) for v in fila if v not in (None, "")]
        if any(n.startswith("id") for n in nombres) and any(n.startswith("bcs") for n in nombres):
            hoja, fila_enc = ws, i
            break
    if hoja is not None:
        break
nombre_hoja = hoja.title
filas = list(hoja.iter_rows(min_row=fila_enc, values_only=True))
wb.close()

encabezado = [texto(c) or f"(columna {i + 1})" for i, c in enumerate(filas[0])]
datos = []
for num, fila in enumerate(filas[1:], start=fila_enc + 1):
    valores = [texto(c) for c in fila][:len(encabezado)]
    if any(valores):
        datos.append([num] + valores + [""] * (len(encabezado) - len(valores)))
bit_raw = pd.DataFrame(datos, columns=["fila_excel"] + encabezado)


def columna(df, *prefijos):
    for c in df.columns:
        if c != "fila_excel" and any(norm(c).startswith(p) for p in prefijos):
            return c
    raise KeyError(f"No encontré una columna que empiece con {prefijos}")


c_id, c_hora, c_bcs = columna(bit_raw, "idvaca", "id"), columna(bit_raw, "hora"), columna(bit_raw, "bcs")
bit = pd.DataFrame({
    "fila_excel": bit_raw["fila_excel"],
    "id_vaca": bit_raw[c_id].str.replace(r"\.0$", "", regex=True),
    "hora_original": bit_raw[c_hora],
    "bcs": pd.to_numeric(bit_raw[c_bcs].str.replace(",", "."), errors="coerce"),
})
print(f"Bitácora: hoja '{nombre_hoja}', encabezados en la fila {fila_enc}, {len(bit)} filas")
print("Columnas:", [c for c in bit_raw.columns if c != "fila_excel"])

**Resultados:** La bitácora está en la hoja [COMPLETAR], con los encabezados en la fila [COMPLETAR]. Tiene [COMPLETAR] filas y las columnas [COMPLETAR].

### Dataset VMS 3

No sabemos con qué programa se exportó `vms-3.csv`, así que **probamos primero la codificación UTF-8 y después Latin-1, y dejamos que pandas detecte el separador**. Leemos todo como texto para no perder formatos.

El ID de la vaca está en la columna "Número del animal". No usamos "Número(s) de selección de animal", que es otra cosa. Si la columna del ID no aparece, se toma la primera.

In [ ]:
# Dataset VMS 3: se prueba la codificación y se detecta el separador
def leer_csv(ruta):
    for enc in ("utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(ruta, sep=None, engine="python", encoding=enc,
                               dtype=str, keep_default_na=False)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"No pude leer {ruta}")


vms = leer_csv(VMS3)
vms.columns = [str(c).strip() for c in vms.columns]
columnas_vms = list(vms.columns)
c_vms_id = next((c for c in columnas_vms if norm(c).startswith("numerodelanimal")), columnas_vms[0])
c_evento = next((c for c in columnas_vms if "eventobcs" in norm(c)), None)
vms["id_vaca"] = vms[c_vms_id].str.strip().str.replace(r"\.0$", "", regex=True)
print(f"vms-3.csv: {len(vms)} filas y {len(columnas_vms)} columnas")
print(f"ID de la vaca: '{c_vms_id}' | Evento BCS: '{c_evento}'")
print(f"CSV de ordeño: {len(csv_ordeno)} vacas")

**Resultados:** `vms-3.csv` tiene [COMPLETAR] filas (una por vaca) y [COMPLETAR] columnas.

**Observaciones:** [COMPLETAR]

## 2.2 Descripción de los datos

### Bitácora de expertos

Resumimos cuántas evaluaciones y vacas tiene la bitácora, y la estadística básica del BCS.

La hora se clasifica en tres formatos: **"HH MM"**, el esperado (por ejemplo "13 00"); **"x"**, cuando no se registró; y **"otro formato"**, para cualquier otra cosa.

In [ ]:
hora = bit["hora_original"].str.strip()
formato_hora = pd.Series(np.select(
    [hora.str.lower().eq("x"), hora.str.match(r"^\d{1,2}\s\d{2}$")],
    ["x (sin dato)", "HH MM"], "otro formato"), index=bit.index)

resumen_bit = pd.Series({
    "evaluaciones (filas)": len(bit),
    "vacas distintas": bit["id_vaca"].nunique(),
    "vacas con más de una evaluación": int((bit["id_vaca"].value_counts() > 1).sum()),
    "BCS mínimo": bit["bcs"].min(),
    "BCS máximo": bit["bcs"].max(),
    "BCS promedio": round(bit["bcs"].mean(), 3),
    "BCS mediana": bit["bcs"].median(),
    "BCS desviación estándar": round(bit["bcs"].std(), 3),
    "todos en pasos de 0.25": bool(((bit["bcs"].dropna() * 4) % 1 == 0).all()),
}, dtype=object)
display(resumen_bit.to_frame("valor"))
display(formato_hora.value_counts().rename_axis("formato de la hora").to_frame("filas"))

**Resultados:** Hay [COMPLETAR] evaluaciones de BCS de [COMPLETAR] vacas distintas; [COMPLETAR] vacas tienen más de una evaluación. El BCS va de [COMPLETAR] a [COMPLETAR] en pasos de 0.25, con promedio de [COMPLETAR] y mediana de [COMPLETAR]. La hora está como "HH MM" en [COMPLETAR] filas, como "x" en [COMPLETAR] y con otro formato en [COMPLETAR].

**Observaciones:** [COMPLETAR]

### Fotografías

**Una foto "única" es un contenido distinto**: si dos archivos tienen la misma huella, cuentan como una sola foto, aunque estén en carpetas distintas o tengan otro nombre. Por eso reportamos el total con copias y sin copias.

La cámara, la resolución y la fecha salen de los metadatos EXIF de cada foto. "Mapa de profundidad" indica si el HEIC trae la información del sensor LiDAR del iPhone.

In [ ]:
unicas = fotos.drop_duplicates("sha256")
resumen_fotos = pd.Series({
    "archivos": len(fotos),
    "fotos únicas": len(unicas),
    "tamaño total (GB)": round(fotos["bytes"].sum() / 1e9, 2),
    "tamaño sin copias (GB)": round(unicas["bytes"].sum() / 1e9, 2),
    "carpetas descargadas": fotos[["id_vaca", "descarga"]].drop_duplicates().shape[0],
    "vacas (IDs de carpeta)": fotos["id_vaca"].nunique(),
    "fotos con mapa de profundidad": int(unicas["profundidad"].sum()),
    "fotos que no se pudieron leer": int((unicas["error"] != "").sum()),
}, dtype=object)
display(resumen_fotos.to_frame("valor"))

display(unicas["ext"].value_counts().rename_axis("formato").to_frame("fotos únicas"))
display(unicas["camara"].replace("", "(sin dato)").value_counts().rename_axis("cámara").to_frame("fotos únicas"))
resolucion = unicas["ancho"].astype("Int64").astype(str) + " × " + unicas["alto"].astype("Int64").astype(str)
display(resolucion.value_counts().rename_axis("resolución (px)").to_frame("fotos únicas"))
display(unicas["fecha"].replace("", "(sin dato)").value_counts().rename_axis("fecha de captura").to_frame("fotos únicas"))

**Resultados:** Hay [COMPLETAR] archivos ([COMPLETAR] GB) y [COMPLETAR] fotos únicas ([COMPLETAR] GB) de [COMPLETAR] vacas. Por formato: [COMPLETAR]. Se tomaron con [COMPLETAR], en [COMPLETAR] resoluciones distintas, el [COMPLETAR]. [COMPLETAR] fotos traen mapa de profundidad.

**Observaciones:** [COMPLETAR]

### Fotos por vaca

Contamos las fotos únicas dentro de cada vaca. **Si la misma foto aparece en dos vacas, cuenta en las dos**; ese caso se revisa en 2.4.

In [ ]:
por_vaca = fotos.drop_duplicates(["id_vaca", "sha256"]).groupby("id_vaca").size().rename("fotos únicas")
por_vaca = por_vaca.loc[ordenar_ids(por_vaca.index)]
display(por_vaca.describe().round(2).to_frame("fotos únicas por vaca"))
display(por_vaca.to_frame())

**Resultados:** Cada vaca tiene entre [COMPLETAR] y [COMPLETAR] fotos únicas, con una mediana de [COMPLETAR].

**Observaciones:** [COMPLETAR]

## 2.3 Exploración de los datos

### Distribución del BCS

Decisiones:
- **Contamos evaluaciones, no vacas**: una vaca evaluada dos veces aporta dos valores. Así el histograma refleja todos los BCS que dieron los expertos.
- **Mostramos todas las clases de 0.25 entre el mínimo y el máximo**, aunque alguna tenga cero evaluaciones, para que se vean los huecos.

Esta distribución importa para las dos métricas del proyecto: con pocas evaluaciones en una clase, la Accuracy puede cambiar mucho con un solo error, y el MAE depende de qué tan lejos quedan las clases extremas del centro.

La gráfica se guarda en `figuras/distribucion_bcs.png`.

In [ ]:
bcs_validos = bit["bcs"].dropna()
clases = np.arange(np.floor(bcs_validos.min() * 4) / 4, bcs_validos.max() + 0.125, 0.25)
conteo = bcs_validos.value_counts().reindex(clases, fill_value=0)
tabla_clases = pd.DataFrame({
    "evaluaciones": conteo.values,
    "porcentaje": (100 * conteo / conteo.sum()).round(1).values,
}, index=pd.Index([f"{c:.2f}" for c in clases], name="BCS"))
display(tabla_clases)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(tabla_clases.index, tabla_clases["evaluaciones"], color="#4C72B0")
for x, y in zip(tabla_clases.index, tabla_clases["evaluaciones"]):
    ax.text(x, y + 0.3, str(y), ha="center", fontsize=9)
ax.set_xlabel("BCS asignado por los expertos")
ax.set_ylabel("Número de evaluaciones")
ax.set_title(f"Distribución del BCS ({len(bcs_validos)} evaluaciones de {bit['id_vaca'].nunique()} vacas)")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIGURAS / "distribucion_bcs.png", dpi=200)
plt.show()

**Resultados:** La clase más frecuente es [COMPLETAR], con [COMPLETAR] evaluaciones ([COMPLETAR]%). Las clases con menos evaluaciones son [COMPLETAR].

**Observaciones:** [COMPLETAR]

### Vacas con más de una evaluación

A algunas vacas las evaluaron dos veces: dos expertos distintos, o la misma vaca en otro momento. **La diferencia entre las dos evaluaciones sirve como referencia de cuánto varía el BCS cuando lo asignan personas.**

Las evaluaciones se ordenan según la fila de la bitácora. Ese orden no indica qué experto dio cada valor.

In [ ]:
rep = bit[bit.duplicated("id_vaca", keep=False)].sort_values(["id_vaca", "fila_excel"])
rep = rep.assign(orden=rep.groupby("id_vaca").cumcount() + 1)
pares = rep.pivot(index="id_vaca", columns="orden", values="bcs")
pares.columns = [f"bcs_{c}" for c in pares.columns]
pares["diferencia"] = pares.max(axis=1) - pares.min(axis=1)
pares = pares.loc[ordenar_ids(pares.index)]
display(pares)
dif = pares["diferencia"]
print(f"{len(pares)} vacas con más de una evaluación | diferencia media {dif.mean():.3f}, "
      f"máxima {dif.max():.2f}, idénticas {(dif == 0).sum()}")

**Resultados:** [COMPLETAR] vacas tienen más de una evaluación. La diferencia entre ellas es de [COMPLETAR] en promedio y de [COMPLETAR] como máximo; en [COMPLETAR] vacas las dos evaluaciones son iguales.

**Observaciones:** [COMPLETAR]

### Fotos por vaca (gráfica)

Histograma del número de fotos únicas de cada vaca. Sirve para ver si unas pocas vacas concentran la mayoría de las fotos. Se guarda en `figuras/fotos_por_vaca.png`.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(por_vaca, bins=range(1, por_vaca.max() + 2), align="left", rwidth=0.85, color="#4C72B0")
ax.set_xticks(range(1, por_vaca.max() + 1))
ax.set_xlabel("Fotos únicas por vaca")
ax.set_ylabel("Número de vacas")
ax.set_title(f"Fotos por vaca ({len(por_vaca)} vacas)")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIGURAS / "fotos_por_vaca.png", dpi=200)
plt.show()

**Resultados:** [COMPLETAR]

### Cruce entre fuentes

Comparamos los IDs de las cuatro fuentes: bitácora, carpetas de fotos, `vms-3.csv` y CSV de ordeño. Una vaca está "en ordeño" si existe un CSV con su ID. **El cruce indica con cuántas vacas se pueden combinar las fotos y el BCS con las otras fuentes.**

In [ ]:
ids_bit, ids_fotos, ids_vms = set(bit["id_vaca"]), set(fotos["id_vaca"]), set(vms["id_vaca"])
conteos = pd.Series({
    "en la bitácora": len(ids_bit),
    "bitácora y fotos": len(ids_bit & ids_fotos),
    "bitácora y vms-3": len(ids_bit & ids_vms),
    "bitácora y ordeño": len(ids_bit & ids_ordeno),
    "bitácora, vms-3 y ordeño": len(ids_bit & ids_vms & ids_ordeno),
    "en vms-3 sin BCS de expertos": len(ids_vms - ids_bit),
})
display(conteos.to_frame("vacas"))
print("Bitácora y ordeño:", ordenar_ids(ids_bit & ids_ordeno))
print("Bitácora, vms-3 y ordeño:", ordenar_ids(ids_bit & ids_vms & ids_ordeno))
print("En vms-3 sin BCS de expertos:", ordenar_ids(ids_vms - ids_bit))

cruce = pd.DataFrame({"id_vaca": ordenar_ids(ids_bit | ids_fotos | ids_vms)})
for nombre, ids in (("bitácora", ids_bit), ("fotos", ids_fotos), ("vms-3", ids_vms), ("ordeño", ids_ordeno)):
    cruce[nombre] = np.where(cruce["id_vaca"].isin(ids), "Sí", "No")
display(cruce)

**Resultados:** De las [COMPLETAR] vacas de la bitácora, [COMPLETAR] tienen fotos, [COMPLETAR] están en `vms-3.csv` y [COMPLETAR] tienen datos de ordeño ([COMPLETAR]). [COMPLETAR] vacas aparecen en la bitácora, en `vms-3.csv` y en ordeño: [COMPLETAR]. En `vms-3.csv` hay [COMPLETAR] vacas sin BCS de los expertos.

**Observaciones:** [COMPLETAR]

### "Evento BCS" del dataset contra el BCS de los expertos

Todavía no sabemos qué mide la columna "Evento BCS" de `vms-3.csv`, así que la comparamos con el BCS de los expertos para ver si coinciden.

Decisiones:
- **Usamos el promedio de las evaluaciones de cada vaca**, porque "Evento BCS" tiene un solo valor por vaca.
- La diferencia es Evento BCS menos el BCS de los expertos: es positiva cuando el dataset da un valor mayor.

In [ ]:
if c_evento is None:
    print("vms-3.csv no tiene una columna 'Evento BCS'")
else:
    evento = vms[["id_vaca", c_evento]].rename(columns={c_evento: "evento_bcs"})
    evento["evento_bcs"] = pd.to_numeric(evento["evento_bcs"].str.replace(",", "."), errors="coerce")
    expertos = bit.groupby("id_vaca")["bcs"].agg(bcs_expertos="mean", evaluaciones="size").reset_index()
    comp = expertos.merge(evento.dropna(subset=["evento_bcs"]), on="id_vaca")
    comp["diferencia"] = (comp["evento_bcs"] - comp["bcs_expertos"]).round(2)
    display(comp.sort_values("diferencia", ascending=False).reset_index(drop=True))
    print(f"{len(comp)} vacas con ambos valores: Evento BCS mayor en {(comp['diferencia'] > 0).sum()}, "
          f"igual en {(comp['diferencia'] == 0).sum()} y menor en {(comp['diferencia'] < 0).sum()}")

**Resultados:** De [COMPLETAR] vacas con ambos valores, Evento BCS es mayor en [COMPLETAR], igual en [COMPLETAR] y menor en [COMPLETAR]. La mayor diferencia es la de la vaca [COMPLETAR] ([COMPLETAR]).

**Observaciones:** [COMPLETAR]

## 2.4 Verificación de la calidad de los datos

Buscamos problemas en los datos tal como llegaron. **Aquí solo se detectan y se miden; cómo se corrigen se decide en la fase 3.**

### Archivos repetidos

- **Copias dentro de una vaca:** archivos con la misma huella en la misma vaca. Pasa cuando una carpeta se descargó dos veces o cuando hay archivos tipo `IMG_0530(1).HEIC`.
- **Vacas con más de una carpeta:** si sus carpetas tienen las mismas fotos, es una descarga repetida; si tienen fotos distintas, no se puede saber qué fotos corresponden a cada evaluación.

In [ ]:
# Copias exactas y descargas repetidas
copias = len(fotos) - len(fotos.drop_duplicates(["id_vaca", "sha256"]))
numeradas = int(fotos["archivo"].str.contains(r"\(\d+\)\.", regex=True).sum())
descargas = fotos.groupby(["id_vaca", "descarga"])["sha256"].apply(frozenset)
dobles = {v: ("mismo contenido" if len(set(g.values)) == 1 else "contenido distinto")
          for v, g in descargas.groupby(level=0) if len(g) > 1}
print(f"Archivos repetidos dentro de una misma vaca: {copias}")
print(f"Archivos con nombre tipo '(1)': {numeradas}")
print(f"Vacas con más de una carpeta descargada: {len(dobles)}")
display(pd.Series(dobles, dtype=object).rename_axis("id_vaca").to_frame("carpetas"))

**Resultados:** Hay [COMPLETAR] archivos repetidos dentro de una misma vaca y [COMPLETAR] nombres tipo "(1)". [COMPLETAR] vacas tienen más de una carpeta: en [COMPLETAR] el contenido es el mismo y en [COMPLETAR] es distinto ([COMPLETAR]).

**Observaciones:** [COMPLETAR]

### Fotos en más de una vaca

**Una foto idéntica asignada a dos vacas distintas es un error de etiquetado**: no se puede saber a cuál de las dos pertenece.

In [ ]:
# Fotos idénticas asignadas a más de una vaca
vacas_por_foto = (fotos.drop_duplicates(["id_vaca", "sha256"])
                  .groupby("sha256")["id_vaca"].apply(lambda s: ordenar_ids(set(s))))
compartidas = vacas_por_foto[vacas_por_foto.map(len) > 1]
grupos_compartidos = Counter(", ".join(v) for v in compartidas)
print(f"Fotos idénticas en más de una vaca: {len(compartidas)}")
display(pd.Series(grupos_compartidos, dtype=object).rename_axis("vacas").to_frame("fotos"))

**Resultados:** [COMPLETAR] fotos idénticas aparecen en más de una vaca: [COMPLETAR].

**Observaciones:** [COMPLETAR]

### Consistencia de la bitácora

- Cada fila de la bitácora debería tener una carpeta de fotos. Comparamos cuántas filas y cuántas carpetas tiene cada ID, y qué IDs aparecen solo en una de las dos fuentes.
- También revisamos las horas que no siguen el formato "HH MM" y los BCS vacíos o fuera del rango de 1 a 5.

In [ ]:
# Consistencia entre la bitácora y las carpetas de fotos
print("IDs de la bitácora sin carpeta de fotos:", ordenar_ids(ids_bit - ids_fotos))
print("Carpetas de fotos sin registro en la bitácora:", ordenar_ids(ids_fotos - ids_bit))

filas_por_id = bit["id_vaca"].value_counts()
carpetas_por_id = fotos[["id_vaca", "descarga"]].drop_duplicates()["id_vaca"].value_counts()
todos = filas_por_id.index.union(carpetas_por_id.index)
consistencia = pd.DataFrame({"filas en la bitácora": filas_por_id.reindex(todos, fill_value=0),
                             "carpetas de fotos": carpetas_por_id.reindex(todos, fill_value=0)})
distintas = consistencia[consistencia.iloc[:, 0] != consistencia.iloc[:, 1]]
print(f"IDs donde no coinciden filas y carpetas: {len(distintas)}")
display(distintas)

# Hora y BCS en la bitácora
print(f"Horas con 'x': {(formato_hora == 'x (sin dato)').sum()} | con otro formato: {(formato_hora == 'otro formato').sum()}")
display(bit.loc[formato_hora == "otro formato", ["fila_excel", "id_vaca", "hora_original"]])
fuera = bit[bit["bcs"].isna() | (bit["bcs"] < 1) | (bit["bcs"] > 5)]
print(f"BCS vacíos o fuera de 1 a 5: {len(fuera)}")

**Resultados:** [COMPLETAR] IDs de la bitácora no tienen carpeta y [COMPLETAR] carpetas no tienen registro en la bitácora. En [COMPLETAR] IDs no coinciden filas y carpetas. Hay [COMPLETAR] horas con "x", [COMPLETAR] con otro formato (filas [COMPLETAR]) y [COMPLETAR] BCS vacíos o fuera de rango.

**Observaciones:** [COMPLETAR]

### Orientación de las fotos

Algunos teléfonos no giran la imagen al guardarla: anotan en los metadatos cómo debe girarse (etiqueta de orientación EXIF). **Si un programa ignora esa etiqueta, la foto se ve de lado**, y muchas librerías de entrenamiento la ignoran.

**Solo revisamos los JPG**, porque en los HEIC la rotación va dentro del formato y la librería la aplica al abrirlos.

In [ ]:
# Orientación: los JPG con orientación EXIF distinta de 1 se ven girados si el programa no la aplica
# (en los HEIC, la rotación va dentro del formato y la librería la aplica al abrirlos)
jpg = unicas[unicas["ext"].isin([".jpg", ".jpeg"])]
giradas = jpg[jpg["orientacion"].fillna(1).astype(int) != 1]
print(f"Fotos JPG: {len(jpg)} | con orientación EXIF distinta de 1: {len(giradas)}")
display(giradas.groupby(giradas["camara"].replace("", "(sin dato)")).size().rename_axis("cámara").to_frame("fotos giradas"))

**Resultados:** De [COMPLETAR] fotos JPG, [COMPLETAR] tienen una orientación EXIF distinta de 1, todas tomadas con [COMPLETAR].

**Observaciones:** [COMPLETAR]

### Resumen de calidad

Juntamos los hallazgos en una tabla con el aspecto, la evidencia y el paso de la fase 3 donde se trata cada uno.

In [ ]:
nombres_compartidas = "; ".join(f"{k} ({v})" for k, v in grupos_compartidos.items()) or "ninguna"
con_distinto = ", ".join(k for k, v in dobles.items() if v == "contenido distinto") or "ninguna"
calidad = pd.DataFrame([
    ["Archivos repetidos", f"{copias} copias; {len(dobles)} vacas con más de una carpeta; {numeradas} nombres tipo '(1)'",
     "3.2 Limpiar: dejar una sola copia por contenido"],
    ["Fotos en más de una vaca", f"{len(compartidas)} fotos idénticas: {nombres_compartidas}",
     "3.2 Limpiar: quitarlas, porque no se sabe a qué vaca pertenecen"],
    ["Carpetas con contenido distinto para una vaca", con_distinto,
     "3.2 Limpiar: no se puede saber qué fotos van con cada evaluación"],
    ["Formatos y resoluciones mezclados",
     f"{', '.join(f'{k} ({v})' for k, v in unicas['ext'].value_counts().items())}; {resolucion.nunique()} resoluciones",
     "3.5 Formatear: un solo formato antes de entrenar"],
    ["Fotos giradas", f"{len(giradas)} JPG con orientación EXIF distinta de 1",
     "3.5 Formatear: enderezarlas al convertir"],
    ["Hora en la bitácora", f"{(formato_hora == 'x (sin dato)').sum()} con 'x' y {(formato_hora == 'otro formato').sum()} con otro formato",
     "No se usa en el modelo; se conserva como registro"],
    ["Vacas con más de una evaluación", f"{len(pares)} vacas; diferencia media {dif.mean():.3f} y máxima {dif.max():.2f}",
     "3.3 Construir: definir la etiqueta de entrenamiento"],
    ["Vista de cada foto", "No viene en los nombres ni en los metadatos",
     "3.3 Construir: marcar la vista de cada foto"],
], columns=["aspecto", "evidencia", "tratamiento (fase 3)"])
with pd.option_context("display.max_colwidth", None):
    display(calidad)

## Conclusión

[COMPLETAR]